# Posture Detection V2 — 08 Hybrid Learned + MotionBERT Webcam

Combines the participant-tested MultiPosture classifier with asynchronous MotionBERT. The learned model handles upright/trunk direction; calibrated MotionBERT supplies the continuous score and remains the fallback when public-data inputs are out of distribution. Press **Q** to save/quit and **C** to recalibrate.

## 1. Load the reusable MotionBERT functions from Notebook 06

In [1]:
import json
from pathlib import Path
wd=Path.cwd(); PROJECT_ROOT=wd.parent if wd.name=='notebooks' else wd if wd.name=='project_v2' else wd/'project_v2'
candidates=[PROJECT_ROOT/'posture_v2_06_realtime_webcam.ipynb',PROJECT_ROOT/'notebooks'/'posture_v2_06_realtime_webcam.ipynb']
NB6=next((p for p in candidates if p.exists()),None)
if NB6 is None: raise FileNotFoundError('Place posture_v2_06_realtime_webcam.ipynb in project_v2 or project_v2/notebooks.')
nb6=json.loads(NB6.read_text(encoding='utf-8'))
# Execute only setup and function-definition cells; the Notebook 06 webcam loop is not executed.
for index in (2,4): exec(''.join(nb6['cells'][index]['source']),globals())
print('Reused MotionBERT setup from:',NB6)

Device: cpu
Reused MotionBERT setup from: c:\Users\manth\Desktop\project_v2\posture_v2_06_realtime_webcam.ipynb


## 2. Load and verify the 99-feature public-data classifier

In [2]:
import joblib, pandas as pd, numpy as np
RF_FILE=PROJECT_ROOT/'models'/'multiposture_upper_body_rf.joblib'; CONTRACT_FILE=PROJECT_ROOT/'models'/'multiposture_upper_body_contract.json'; PUBLIC_CSV=PROJECT_ROOT/'data'/'public_multiposture'/'data.csv'
for p in (RF_FILE,CONTRACT_FILE,PUBLIC_CSV):
    if not p.exists(): raise FileNotFoundError(f'Run Notebook 07 first: {p}')
rf=joblib.load(RF_FILE); contract=json.loads(CONTRACT_FILE.read_text()); FEATURE_COLUMNS=contract['feature_columns']
if hasattr(rf,'n_jobs'): rf.n_jobs=1  # Avoid repeated joblib/threading warnings during webcam inference
LANDMARK_NAMES=['nose','left_eye_inner','left_eye','left_eye_outer','right_eye_inner','right_eye','right_eye_outer','left_ear','right_ear','mouth_left','mouth_right','left_shoulder','right_shoulder','left_elbow','right_elbow','left_wrist','right_wrist','left_pinky','right_pinky','left_index','right_index','left_thumb','right_thumb','left_hip','right_hip','left_knee','right_knee','left_ankle','right_ankle','left_heel','right_heel','left_foot_index','right_foot_index']
expected=[f'{name}_{axis}' for name in LANDMARK_NAMES for axis in 'xyz']
if set(FEATURE_COLUMNS)!=set(expected) or len(FEATURE_COLUMNS)!=99: raise ValueError('Saved feature contract is not the expected MediaPipe-33 x/y/z schema.')
public=pd.read_csv(PUBLIC_CSV,usecols=FEATURE_COLUMNS); lower=public.quantile(.005); upper=public.quantile(.995)
def public_features(landmarks):
    xyz=np.array([[p.x,p.y,p.z] for p in landmarks],np.float32); hip=(xyz[23]+xyz[24])/2; xyz-=hip
    values={f'{name}_{axis}':float(xyz[i,j]) for i,name in enumerate(LANDMARK_NAMES) for j,axis in enumerate('xyz')}
    return pd.DataFrame([[values[c] for c in FEATURE_COLUMNS]],columns=FEATURE_COLUMNS)
def classify_public(landmarks):
    x=public_features(landmarks); outside=((x.iloc[0]<lower)|(x.iloc[0]>upper)).mean(); probability=rf.predict_proba(x)[0]; i=int(np.argmax(probability))
    return str(rf.classes_[i]),float(probability[i]),float(outside),outside<=.20
print('Classifier classes:',list(rf.classes_)); print('Feature contract verified:',len(FEATURE_COLUMNS))

Classifier classes: ['lean_backward', 'lean_forward', 'lean_left', 'lean_right', 'upright']
Feature contract verified: 99


## 3. Run hybrid webcam

Sit upright during the eight MotionBERT calibration updates. The learned label is shown only when at least 55% confident and no more than 20% of features fall outside public-data ranges.

In [3]:
from collections import Counter,deque
from concurrent.futures import ThreadPoolExecutor
CAMERA_INDEX=0; WINDOW=81; STRIDE=27; CALIBRATION_UPDATES=8; CLASS_SMOOTH=15
Base=mp.tasks.BaseOptions; Landmarker=mp.tasks.vision.PoseLandmarker; Options=mp.tasks.vision.PoseLandmarkerOptions; Mode=mp.tasks.vision.RunningMode
opts=Options(base_options=Base(model_asset_path=str(MODEL_TASK)),running_mode=Mode.VIDEO,num_poses=1,min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,min_tracking_confidence=.5)
cap=cv2.VideoCapture(CAMERA_INDEX); cap.set(cv2.CAP_PROP_FRAME_WIDTH,1280); cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
if not cap.isOpened(): raise RuntimeError('Could not open webcam.')
buffer=deque(maxlen=WINDOW); votes=deque(maxlen=CLASS_SMOOTH); executor=ThreadPoolExecutor(max_workers=1); future=None; calibration=[]; baseline=scale=None; score=None; rule_issue='none'; latency=np.nan; history=[]; frame_id=0; started=time.perf_counter(); learned='waiting'; status='BUFFERING'
label='unknown'; prob=np.nan; ood=np.nan; trusted=False
with Landmarker.create_from_options(opts) as detector:
  try:
    while True:
      ok,frame=cap.read();
      if not ok: break
      frame=cv2.flip(frame,1); h,w=frame.shape[:2]; now=time.perf_counter(); ts=int((now-started)*1000)
      result=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)),ts); detected=bool(result.pose_landmarks)
      if detected:
        landmarks=result.pose_landmarks[0]; buffer.append(to_h36m(landmarks)); label,prob,ood,trusted=classify_public(landmarks)
        if trusted and prob>=.55:
          votes.append(label)
        else:
          votes.clear()  # Never reuse stale predictions after a rejected frame
        learned=Counter(votes).most_common(1)[0][0] if votes else 'uncertain'; learned_info=f'{learned} {prob:.0%} OOD:{ood:.0%}'
      else: buffer.clear(); votes.clear(); learned='low confidence'; learned_info=learned
      if future is not None and future.done():
        pose3d,latency=future.result(); value=features(pose3d)
        if baseline is None:
          calibration.append(value); status=f'CALIBRATING {len(calibration)}/{CALIBRATION_UPDATES}'
          if len(calibration)>=CALIBRATION_UPDATES:
            a=np.stack(calibration); baseline=np.median(a,axis=0); scale=np.maximum(1.4826*np.median(np.abs(a-baseline),axis=0),FLOOR); status='CALIBRATED'
        else:
          score,rule_issue,severity=score_pose(value,baseline,scale); trunk_bad=learned!='upright' and learned not in ('uncertain','low confidence'); status='BAD' if score<70 or trunk_bad else 'GOOD'
          issue=learned if trunk_bad else rule_issue; history.append({'utc':datetime.now(timezone.utc).isoformat(),'seconds':now-started,'score':score,'raw_learned_label':label,'learned_probability':prob,'out_of_distribution_fraction':ood,'public_model_trusted':trusted,'learned_trunk':learned,'rule_issue':rule_issue,'final_issue':issue,'state':status,'motionbert_latency_ms':latency})
        future=None
      if len(buffer)==WINDOW and frame_id%STRIDE==0 and future is None: future=executor.submit(lift,prepare(np.stack(buffer),w,h))
      color=(0,220,0) if status in ('GOOD','CALIBRATED') else (0,200,255) if 'CALIB' in status or 'BUFFER' in status else (0,0,255)
      cv2.putText(frame,status,(25,40),cv2.FONT_HERSHEY_SIMPLEX,.9,color,2); cv2.putText(frame,f'Learned trunk: {learned_info}',(25,78),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
      if score is not None: cv2.putText(frame,f'3D score: {score:.1f}  Rule: {rule_issue}  MB: {latency:.0f}ms',(25,112),cv2.FONT_HERSHEY_SIMPLEX,.62,color,2)
      cv2.putText(frame,'Q quit | C recalibrate',(25,h-25),cv2.FONT_HERSHEY_SIMPLEX,.55,(255,255,255),1); cv2.imshow('Posture V2 Hybrid',frame)
      key=cv2.waitKey(1)&0xFF
      if key==ord('q'): break
      if key==ord('c'): calibration=[]; baseline=scale=None; score=None; votes.clear(); status='RECALIBRATING'
      frame_id+=1
  finally: cap.release(); cv2.destroyAllWindows(); executor.shutdown(wait=True)
session=pd.DataFrame(history); out=OUTPUT_DIR/f"hybrid_webcam_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"; session.to_csv(out,index=False); print('Saved:',out.resolve()); display(session.tail())
if len(session):
  accepted=session['public_model_trusted'] & session['learned_probability'].ge(.55)
  diagnostics=pd.Series({'updates':len(session),'median_probability':session['learned_probability'].median(),'median_ood_fraction':session['out_of_distribution_fraction'].median(),'trusted_percent':session['public_model_trusted'].mean()*100,'accepted_prediction_percent':accepted.mean()*100})
  display(diagnostics.to_frame('value')); display(session['raw_learned_label'].value_counts().to_frame('updates'))

Saved: C:\Users\manth\Desktop\project_v2\results\webcam_sessions\hybrid_webcam_20260924_150042.csv


,utc,seconds,score,raw_learned_label,learned_probability,out_of_distribution_fraction,public_model_trusted,learned_trunk,rule_issue,final_issue,state,motionbert_latency_ms
24,2026-09-24T09:30:29.413066+00:00,94.034002,30.000000,lean_left,0.669847,0.292929,False,uncertain,slouching,slouching,BAD,1261.1069
25,2026-09-24T09:30:31.928882+00:00,96.522052,65.000000,lean_forward,0.708753,0.454545,False,uncertain,leaning,leaning,BAD,1285.2450
26,2026-09-24T09:30:34.566971+00:00,99.172520,100.000000,lean_forward,0.591950,0.282828,False,uncertain,none,none,GOOD,1423.1962
27,2026-09-24T09:30:37.015114+00:00,101.598477,80.736509,lean_right,0.548076,0.393939,False,uncertain,shoulder imbalance,shoulder imbalance,GOOD,1313.5065
28,2026-09-24T09:30:39.628200+00:00,104.246256,45.760147,lean_right,0.556089,0.353535,False,uncertain,shoulder imbalance,shoulder imbalance,BAD,1353.6306


,value
updates,29.000000
median_probability,0.700650
median_ood_fraction,0.393939
trusted_percent,0.000000
accepted_prediction_percent,0.000000


,updates
raw_learned_label,
lean_forward,18
lean_left,6
lean_right,5


## Interpretation

The public classifier is accepted only after schema, probability, and range checks. Treat this as external-domain validation: the 96.36% test accuracy was measured on held-out MultiPosture participants, not on your webcam. Collecting a small labelled webcam adaptation set would be required to quantify local accuracy.